# Finetuning on NLI

## Setup & Load Data

In [1]:
!nvidia-smi

Sun Jan  9 21:55:44 2022       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 496.76       Driver Version: 496.76       CUDA Version: 11.5     |
|-------------------------------+----------------------+----------------------+
| GPU  Name            TCC/WDDM | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA GeForce ... WDDM  | 00000000:01:00.0  On |                  N/A |
|  0%   49C    P8    14W / 200W |   2613MiB /  8192MiB |      2%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [2]:
import torch
print(torch.cuda.is_available())

True


In [3]:
from datasets import load_dataset

train_dataset = load_dataset("newsph_nli", split='train+validation+test')
#test_dataset = load_dataset("newsph_nli", split='test') # TEMPORARILY RENAME to train_dataset to subset

print(train_dataset)
#print("================================================")
#print(test_dataset)

Using custom data configuration default
Reusing dataset newsph_nli (C:\Users\CCS\.cache\huggingface\datasets\newsph_nli\default\1.0.0\696e8afaafaffcd18a9c2144c3f2bb14c0d138397339252be02e395493766145)


Dataset({
    features: ['premise', 'hypothesis', 'label'],
    num_rows: 519000
})


In [4]:
negatives = 0
for data in train_dataset:
    negatives += data['label']

In [5]:
positives = len(train_dataset) - negatives
print(f"POSITIVE PAIRS: {positives}")
print(f"NEGATIVE PAIRS: {negatives}")

POSITIVE PAIRS: 237679
NEGATIVE PAIRS: 281321


## Prepare data

In [4]:
print(f"before: {len(train_dataset)} rows")
train_dataset = train_dataset.filter(
    lambda x: True if x['label'] == 0 else False
)
print(f"after: {len(train_dataset)} rows")

Loading cached processed dataset at C:\Users\CCS\.cache\huggingface\datasets\newsph_nli\default\1.0.0\696e8afaafaffcd18a9c2144c3f2bb14c0d138397339252be02e395493766145\cache-71f78a27c97499c2.arrow


before: 519000 rows
after: 237679 rows


In [5]:
from sentence_transformers import InputExample
from tqdm.auto import tqdm  # so we see progress bar

train_samples = []
for row in tqdm(train_dataset):
    train_samples.append(InputExample(
        texts=[row['premise'], row['hypothesis']]
    ))

  0%|          | 0/237679 [00:00<?, ?it/s]

In [6]:
from sentence_transformers import datasets

batch_size = 16

loader = datasets.NoDuplicatesDataLoader(
    train_samples, batch_size=batch_size)

## Prepare model

In [7]:
MODEL_DIR = r"D:/thesis/model"

In [8]:
from sentence_transformers import models, SentenceTransformer
from sentence_transformers import losses

def model_loss_init():
    roberta = models.Transformer(fr"{MODEL_DIR}/checkpoint-990000", max_seq_length=64)
    print(f"LOADED MODEL: {roberta}")

    pooler = models.Pooling(
        roberta.get_word_embedding_dimension(),
        pooling_mode_mean_tokens=True
    )

    model = SentenceTransformer(modules=[roberta, pooler])
    loss = losses.MultipleNegativesRankingLoss(model)

    return model, loss

## Training (default hyperparameters from author)

In [13]:
epochs = 1
warmup_steps = int(len(loader) * epochs * 0.1)
warmup_steps

1485

In [14]:
%%time

epochs = 1
warmup_steps = int(len(loader) * epochs * 0.1)

model, loss = model_loss_init()

print(f"MODEL FOR FINETUNING: {model}")
print(loss)

model.fit(
    train_objectives=[(loader, loss)],
    epochs=epochs,
    warmup_steps=warmup_steps,
    output_path='./sroberta-nli-1epoch-v2',
    show_progress_bar=True,
    use_amp=True
)

Some weights of the model checkpoint at D:/thesis/model/checkpoint-990000 were not used when initializing RobertaModel: ['lm_head.dense.bias', 'lm_head.layer_norm.weight', 'lm_head.dense.weight', 'lm_head.bias', 'lm_head.layer_norm.bias']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaModel were not initialized from the model checkpoint at D:/thesis/model/checkpoint-990000 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and i

LOADED MODEL: Transformer({'max_seq_length': 64, 'do_lower_case': False}) with Transformer model: RobertaModel 
MODEL FOR FINETUNING: SentenceTransformer(
  (0): Transformer({'max_seq_length': 64, 'do_lower_case': False}) with Transformer model: RobertaModel 
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False})
)
MultipleNegativesRankingLoss(
  (model): SentenceTransformer(
    (0): Transformer({'max_seq_length': 64, 'do_lower_case': False}) with Transformer model: RobertaModel 
    (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False})
  )
  (cross_entropy_loss): CrossEntropyLoss()
)


Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Iteration:   0%|          | 0/3713 [00:00<?, ?it/s]

Wall time: 19min 14s


## Evaluate

In [21]:
from sentence_transformers import util

emb1 = model.encode("Kumalat ang apoy sa kagubatan.")
emb2 = model.encode("Lumalakas ang apoy ng aking damdamin.")
emb2 = model.encode("Bumuga ng apoy ang dragon.")

cos_sim = util.cos_sim(emb1, emb2)
print("Cosine-Similarity:", cos_sim)

Cosine-Similarity: tensor([[0.8643]])


In [13]:
# JANUARY 3, 2022, 16 bs 780k model
from sentence_transformers import util

emb1 = model.encode("Kumalat ang apoy sa kagubatan.")
emb2 = model.encode("Lumalakas ang apoy ng aking damdamin.")
emb2 = model.encode("Bumuga ng apoy ang dragon.")

cos_sim = util.cos_sim(emb1, emb2)
print("Cosine-Similarity:", cos_sim)

Cosine-Similarity: tensor([[0.8612]])


In [21]:
# JANUARY 3, 2022, 16 bs 990k model
from sentence_transformers import util

emb1 = model.encode("Kumalat ang apoy sa kagubatan.")
emb2 = model.encode("Lumalakas ang apoy ng aking damdamin.")
emb2 = model.encode("Bumuga ng apoy ang dragon.")

cos_sim = util.cos_sim(emb1, emb2)
print("Cosine-Similarity:", cos_sim)

Cosine-Similarity: tensor([[0.8554]])


In [10]:
# JANUARY 3, 2022, 32 bs 64 msl 990k model
from sentence_transformers import util

emb1 = model.encode("Kumalat ang apoy sa kagubatan.")
emb2 = model.encode("Lumalakas ang apoy ng aking damdamin.")
emb2 = model.encode("Bumuga ng apoy ang dragon.")

cos_sim = util.cos_sim(emb1, emb2)
print("Cosine-Similarity:", cos_sim)

Cosine-Similarity: tensor([[0.8543]])


In [15]:
# JANUARY 3, 2022, 32 bs 64 msl 990k model
from sentence_transformers import util

emb1 = model.encode("Kumalat ang apoy sa kagubatan.")
emb2 = model.encode("Lumalakas ang apoy ng aking damdamin.")
emb2 = model.encode("Bumuga ng apoy ang dragon.")

cos_sim = util.cos_sim(emb1, emb2)
print("Cosine-Similarity:", cos_sim)

Cosine-Similarity: tensor([[0.8192]])


## Clear GPU memory

In [47]:
torch.cuda.empty_cache()